# Train Arm B at 500 steps / 0.2GB (fair 3-way comparison, question 1.3)

See `plans/PLAN.md` question 1.3, `phases/phase-2-small-train.md`. Same
dataset_target_gb and max_steps as the matching Arm A/C notebooks.

**Rerun** after fixing `entropy_threshold` (3.7 -> 3.0, see phase doc "Phát hiện: entropy_threshold
bị hiệu chỉnh sai") and bumping `entropy_pretrain_steps` 100 -> 500 to match the value
the threshold was actually calibrated against (and what the 5000/20000-step runs used).


In [ ]:
import subprocess

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()
print("GPU:", gpu_name or "(none detected)")

if "P100" in gpu_name:
    print("P100 detected - pinning torch==2.7.1+cu126 (last version supporting sm_60)")
    subprocess.run(
        ["pip", "install", "-q", "torch==2.7.1", "--index-url",
         "https://download.pytorch.org/whl/cu126"],
        check=True,
    )
else:
    print("Not a P100 - keeping the pre-installed torch build")

import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())


In [ ]:
import os

_candidates = [
    "/kaggle/input/vislm-research-code",
    "/kaggle/input/datasets/nguyennn263/vislm-research-code",
]
CODE_DIR = next(p for p in _candidates if os.path.isdir(p))
os.environ["PYTHONPATH"] = CODE_DIR
print("CODE_DIR:", CODE_DIR)


In [ ]:
!pip install -q "transformers==4.46.3" datasets pyyaml


In [ ]:
!python {CODE_DIR}/setup/download_prepare_data.py \
  --target-gb 0.2 --out-dir /kaggle/working/data/prepared/fineweb2_vi --shard-size-mb 50


In [ ]:
!python -m vislm.train {CODE_DIR}/pillar1_configs/1_3_arm_B_blt.yaml \
  dataset=/kaggle/working/data/prepared/fineweb2_vi \
  run_dir=/kaggle/working/runs/arm_B_500 \
  train.max_steps=500 train.entropy_pretrain_steps=500


In [ ]:
import json

losses = []
with open("/kaggle/working/runs/arm_B_500/metrics.jsonl") as f:
    for line in f:
        row = json.loads(line)
        if "step" in row:
            losses.append(row["loss"])

summary = {
    "n_steps": len(losses),
    "first_loss": losses[0],
    "last_loss": losses[-1],
    "min_loss": min(losses),
    "last_20_avg": sum(losses[-20:]) / len(losses[-20:]),
}
print(summary)

with open("/kaggle/working/metrics_train_arm_b_500.jsonl", "w") as f:
    f.write(json.dumps({"section": "train_arm_b_500", "results": summary}) + "\n")
